
RealHedge0Chain.py
--------------
A simplified blockchain (Task 1) using Proof-of-Authority (PoA) consensus
with REAL cryptographic validator signatures (ECDSA over the secp256k1
curve - the same curve Ethereum itself uses).

Design choices, matched to the RealHedge use case:
    - PoA rather than PoW: RealHedge is a permissioned, real-estate
      platform. A small set of trusted validators (platform operator,
      independent auditor) sign blocks rather than competing to solve
      a computational puzzle. Faster, deterministic finality; no wasted
      energy - appropriate for a regulated-adjacent financial application.
    - Every block stores: timestamp, transaction data, this block's hash,
      the previous block's hash, and a validator signature to seal the block


# Importing libraries

In [1]:
import hashlib
import json
import time

from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec

#1. Validator identity

Represents one validator NODE. In a real deployment each validator would generate and keep this keypair privately on their own machine and only ever publish the public key. Here, for a single-process demo, the object simply holds both.

Note that Blockchain (below) only ever stores the PUBLIC key, and only the PoAValidator object itself can produce signatures, mirroring the real separation.

In [2]:
# ======================================================================
# Validator identity "Class": a real ECDSA keypair
# ======================================================================
class PoAValidator:

    def __init__(self, validator_id: str):
        self.validator_id = validator_id
        self._private_key = ec.generate_private_key(ec.SECP256K1())
        self.public_key = self._private_key.public_key()

    def public_key_hex(self) -> str:
        pub_bytes = self.public_key.public_bytes(
            encoding=serialization.Encoding.X962,
            format=serialization.PublicFormat.UncompressedPoint,
        )
        return pub_bytes.hex()

    def sign(self, message: bytes) -> str:
        signature = self._private_key.sign(message, ec.ECDSA(hashes.SHA256()))
        return signature.hex()

    def __repr__(self):
        return f"PoAValidator(id={self.validator_id}, pubkey={self.public_key_hex()[:16]}...)"

1.1 Verifying the Validators Signature

In [3]:
def verify_signature(public_key_hex: str, message: bytes, signature_hex: str) -> bool:
    try:
        pub_bytes = bytes.fromhex(public_key_hex)
        public_key = ec.EllipticCurvePublicKey.from_encoded_point(ec.SECP256K1(), pub_bytes)
        public_key.verify(bytes.fromhex(signature_hex), message, ec.ECDSA(hashes.SHA256()))
        return True
    except (InvalidSignature, ValueError):
        return False

#2. RealHedge Block

In [4]:
# ======================================================================
# Block
# ======================================================================
class Block:
    def __init__(self, index: int, timestamp: float, transactions: list,
                 previous_hash: str, validator_id: str):
        self.index = index
        self.timestamp = timestamp
        self.transactions = transactions          # list of dicts
        self.previous_hash = previous_hash
        self.validator_id = validator_id           # WHO produced this block
        self.hash = self.compute_hash()             # content hash (unsigned)
        self.signature = None                        # set once the validator signs it

    def compute_hash(self) -> str:

        # Hash covers index/timestamp/transactions/previous_hash/validator_id.
        # Any change to any of those after the fact produces a different
        # hash - this is what both tamper-detection AND signature
        # verification key off.

        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "validator_id": self.validator_id,
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def to_dict(self) -> dict:
        return {
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": self.transactions,
            "previous_hash": self.previous_hash,
            "validator_id": self.validator_id,
            "hash": self.hash,
            "signature": self.signature,
        }

    def __repr__(self):
        return (f"Block(#{self.index}, txs={len(self.transactions)}, "
                f"validator={self.validator_id}, hash={self.hash[:12]}..., "
                f"signed={self.signature is not None})")

# 3. RealHedge Block-Chain

In [5]:
# ======================================================================
# Blockchain
# ======================================================================
class Blockchain:
    def __init__(self):
        self.chain: list = []
        self.pending_transactions: list = []
        # validator_id -> public key hex. Only public keys are ever
        # stored here - the chain can VERIFY signatures but can never
        # PRODUCE one, exactly like a real PoA node set.
        self.validator_public_keys: dict = {}
        self._create_genesis_block()

    # ------------------------------------------------------------------
    # Genesis
    # ------------------------------------------------------------------
    def _create_genesis_block(self):
        genesis = Block(index=0, timestamp=time.time(), transactions=[],
                         previous_hash="0" * 64, validator_id="genesis")
        genesis.signature = "genesis"  # no real signature needed for genesis
        self.chain.append(genesis)

    @property
    def last_block(self) -> Block:
        return self.chain[-1]

    # ------------------------------------------------------------------
    # PoA validator management
    # ------------------------------------------------------------------
    def register_validator(self, validator: PoAValidator):

        # Authorise a validator by recording ONLY their public key
        self.validator_public_keys[validator.validator_id] = validator.public_key_hex()

    def revoke_validator(self, validator_id: str):
        self.validator_public_keys.pop(validator_id, None)

    def is_authorized(self, validator_id: str) -> bool:
        return validator_id in self.validator_public_keys

    # ------------------------------------------------------------------
    # Transactions / mining
    # ------------------------------------------------------------------
    def add_transaction(self, transaction: dict):
        transaction = dict(transaction)
        transaction.setdefault("submitted_at", time.time())
        self.pending_transactions.append(transaction)


    def mine_block(self, validator: PoAValidator) -> Block:

        # PoA consensus with real signing:
        #   1. Validator must be authorized (their public key is registered).
        #   2. The block's content hash is computed.
        #   3. The validator SIGNS that hash with their private key.
        #   4. The block only becomes part of the chain once signed.
        # Any other participant can later verify the signature using only
        # the validator's public key (see is_chain_valid), without ever needing the private key.

        if not self.is_authorized(validator.validator_id):
            raise PermissionError(
                f"'{validator.validator_id}' is not an authorized PoA validator; block rejected."
            )
        registered_key = self.validator_public_keys[validator.validator_id]
        if registered_key != validator.public_key_hex():
            raise PermissionError(
                f"Public key mismatch for '{validator.validator_id}' - possible impersonation."
            )
        if not self.pending_transactions:
            raise ValueError("No pending transactions to mine.")

        new_block = Block(
            index=len(self.chain),
            timestamp=time.time(),
            transactions=self.pending_transactions,
            previous_hash=self.last_block.hash,
            validator_id=validator.validator_id,
        )
        new_block.signature = validator.sign(new_block.hash.encode())

        self.chain.append(new_block)
        self.pending_transactions = []
        return new_block

    # ------------------------------------------------------------------
    # Validation / tamper detection
    # ------------------------------------------------------------------
    def is_chain_valid(self):
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            # 1. Has this block's own content been altered since it was mined?
            if current.hash != current.compute_hash():
                return False, f"Block {current.index} has been tampered with (hash mismatch)."

            # 2. Does this block still correctly point to the previous block?
            if current.previous_hash != previous.hash:
                return False, (f"Chain broken between block {previous.index} and "
                                f"{current.index} (previous_hash link mismatch).")

            # 3. Was this block signed by an authorized validator, and is
            #    that signature actually valid over this block's hash?
            if current.validator_id not in self.validator_public_keys:
                return False, f"Block {current.index} was signed by an unregistered validator."

            public_key_hex = self.validator_public_keys[current.validator_id]
            if not verify_signature(public_key_hex, current.hash.encode(), current.signature or ""):
                return False, (f"Block {current.index}'s signature does not verify against "
                                f"validator '{current.validator_id}''s public key "
                                f"(forged or corrupted signature).")

        return True, "Chain is valid: all hashes link correctly and every signature verifies."

    def print_chain(self):
        for block in self.chain:
            print(f"--- Block {block.index} ---")
            print(f"  Timestamp     : {block.timestamp}")
            print(f"  Validator     : {block.validator_id}")
            print(f"  Previous Hash : {block.previous_hash}")
            print(f"  Hash          : {block.hash}")
            sig_preview = (block.signature[:24] + "...") if block.signature and block.signature != "genesis" else block.signature
            print(f"  Signature     : {sig_preview}")
            print(f"  Transactions  : {len(block.transactions)}")
            for tx in block.transactions:
                print(f"    - {tx}")


# 4. Testing the RealHedge0Chain

  exercises every core behavior of RealHedge0Chain.py:
  1. Genesis block creation
  2. Validator registration (public key only)
  3. Adding transactions + mining a signed block
  4. Chain validity on a clean chain
  5. Tamper detection (data altered without recalculating hash)
  6. Previous-hash link tampering
  7. Unauthorized validator rejected (PermissionError)
  8. Forged signature detection (impostor with a different private key)


In [6]:
def banner(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def show_block(block, label=""):
    print(f"{label}Block #{block.index}")
    print(f"  Timestamp     : {block.timestamp}")
    print(f"  Transactions  : {block.transactions}")
    print(f"  Previous hash : {block.previous_hash}")
    print(f"  Hash          : {block.hash}")
    print(f"  Validator     : {block.validator_id}")
    sig_display = block.signature if block.signature == "genesis" else f"{block.signature[:40]}..."
    print(f"  Signature     : {sig_display}")


4.1 Scenario 1 : Creating te genesis block

In [7]:
# ======================================================================
banner("STEP 1: CREATE THE CHAIN — INSPECT THE GENESIS BLOCK")
# ======================================================================
chain = Blockchain()
print("A new Blockchain() automatically creates block #0, the genesis block.")
print("It has no real transactions and its previous_hash is all zeros,")
print("since there is no block before it.\n")
show_block(chain.chain[0])


STEP 1: CREATE THE CHAIN — INSPECT THE GENESIS BLOCK
A new Blockchain() automatically creates block #0, the genesis block.
It has no real transactions and its previous_hash is all zeros,
since there is no block before it.

Block #0
  Timestamp     : 1788955663.489944
  Transactions  : []
  Previous hash : 0000000000000000000000000000000000000000000000000000000000000000
  Hash          : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Validator     : genesis
  Signature     : genesis


4.2 Scenario 2 : Registering the Validators  (public key only)

In [8]:
# ======================================================================
banner("STEP 2: REGISTER PoA VALIDATORS — REAL ECDSA KEYPAIRS")
# ======================================================================
print("Each validator generates its own private/public keypair on creation.")
print("Only the PUBLIC key is ever given to the chain (register_validator).\n")

validator_a = PoAValidator("validator_A")
validator_b = PoAValidator("validator_B")
validator_c= PoAValidator("validator_C")
chain.register_validator(validator_a)
chain.register_validator(validator_b)
chain.register_validator(validator_c)
print(f"validator_A public key: {validator_a.public_key_hex()}")
print(f"validator_B public key: {validator_b.public_key_hex()}")
print(f"validator_c public key: {validator_c.public_key_hex()}")

print(f"\nWhat the chain actually stores (chain.validator_public_keys):")
print(f"  {chain.validator_public_keys}")

### Notice the chain never sees any validator's PRIVATE key


STEP 2: REGISTER PoA VALIDATORS — REAL ECDSA KEYPAIRS
Each validator generates its own private/public keypair on creation.
Only the PUBLIC key is ever given to the chain (register_validator).

validator_A public key: 0455460516a4cc3935c808fdfeafdd5257b5e58d99efd7da5fabd61f48065dd5138ee8c29f93ea49a4a8c2811e7e2306bea0e141a7c8ab03e228ebb84044dd1fc1
validator_B public key: 04627c9ab893dc5980926455abd4e0ceb5a886761c854286a4b25cc224715f2e4f73638a13a3fc8debc66650e43d8b6a016f9f4ab31ab5f3543ef895d96450cc1f
validator_c public key: 04bdc78d33fe48cdac028ef576e2d7517233b329a01a2109110c61f2adba8feff2e20571ab777827ca020ebf4b41de0eff58da9fc8589e6eaacd8bd7e55dc8c4ad

What the chain actually stores (chain.validator_public_keys):
  {'validator_A': '0455460516a4cc3935c808fdfeafdd5257b5e58d99efd7da5fabd61f48065dd5138ee8c29f93ea49a4a8c2811e7e2306bea0e141a7c8ab03e228ebb84044dd1fc1', 'validator_B': '04627c9ab893dc5980926455abd4e0ceb5a886761c854286a4b25cc224715f2e4f73638a13a3fc8debc66650e43d8b6a016f9f4ab31ab5

4.3 Scenario3 : Making a transaction and mining it

In [9]:
# ======================================================================
banner("STEP 3: ADD A TRANSACTION AND MINE A REAL SIGNED BLOCK")
# ======================================================================
tx1 = {"event": "PROPERTY_CREATED", "property_id": 1,
       "name": "London Apartment", "initial_value": 500_000, "total_shares": 3_000}

print(f"Adding transaction to the mempool (chain.add_transaction):\n  {tx1}\n")
chain.add_transaction(tx1)
print(f"Pending transactions before mining: {chain.pending_transactions}\n")

print("validator_A mines the block — this SIGNS the block's hash with")
print("validator_A's private key, which the chain itself never holds.\n")
block1 = chain.mine_block(validator_a)
show_block(block1, label="Result: ")

print(f"\nPending transactions after mining (should be empty): {chain.pending_transactions}")


STEP 3: ADD A TRANSACTION AND MINE A REAL SIGNED BLOCK
Adding transaction to the mempool (chain.add_transaction):
  {'event': 'PROPERTY_CREATED', 'property_id': 1, 'name': 'London Apartment', 'initial_value': 500000, 'total_shares': 3000}

Pending transactions before mining: [{'event': 'PROPERTY_CREATED', 'property_id': 1, 'name': 'London Apartment', 'initial_value': 500000, 'total_shares': 3000, 'submitted_at': 1788955663.5416627}]

validator_A mines the block — this SIGNS the block's hash with
validator_A's private key, which the chain itself never holds.

Result: Block #1
  Timestamp     : 1788955663.5420365
  Transactions  : [{'event': 'PROPERTY_CREATED', 'property_id': 1, 'name': 'London Apartment', 'initial_value': 500000, 'total_shares': 3000, 'submitted_at': 1788955663.5416627}]
  Previous hash : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Hash          : 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  Validator     : validator_A
  Sig

In [10]:

# ======================================================================
banner("STEP 4: ADD A SECOND BLOCK — SEE THE HASH CHAIN LINK FORM")
# ======================================================================
tx2 = {"event": "SHARE_TRANSFERRED", "property_id": 1,
       "from": "0xAlice", "to": "0xBob", "amount": 100}

print(f"Adding a second transaction:\n  {tx2}\n")
chain.add_transaction(tx2)

print("validator_B mines this one — a DIFFERENT validator, to show the")
print("chain does not require the same validator to seal every block.\n")
block2 = chain.mine_block(validator_b)
show_block(block2, label="Result: ")

print(f"\nCheck the link: does block2.previous_hash match block1.hash?")
print(f"  block2.previous_hash = {block2.previous_hash}")
print(f"  block1.hash          = {block1.hash}")
print(f"  Match: {block2.previous_hash == block1.hash}")



STEP 4: ADD A SECOND BLOCK — SEE THE HASH CHAIN LINK FORM
Adding a second transaction:
  {'event': 'SHARE_TRANSFERRED', 'property_id': 1, 'from': '0xAlice', 'to': '0xBob', 'amount': 100}

validator_B mines this one — a DIFFERENT validator, to show the
chain does not require the same validator to seal every block.

Result: Block #2
  Timestamp     : 1788955663.5717082
  Transactions  : [{'event': 'SHARE_TRANSFERRED', 'property_id': 1, 'from': '0xAlice', 'to': '0xBob', 'amount': 100, 'submitted_at': 1788955663.571571}]
  Previous hash : 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  Hash          : 838eeff7543497d857a856f19bd7cdb31ba7298b21eec872488d2131fc6e6eb8
  Validator     : validator_B
  Signature     : 30440220702e24299d24669633b8cc30c696d6cd...

Check the link: does block2.previous_hash match block1.hash?
  block2.previous_hash = 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  block1.hash          = 986f57eeceb81b6ba36f1defb83271683d181c199

4.4 Scenario 4: checking the validty of the chain , each block is correctly connected to the previous hashes

In [11]:
# ======================================================================
banner("STEP 5: VALIDATE THE WHOLE CHAIN")
# ======================================================================
valid, msg = chain.is_chain_valid()
print(f"is_chain_valid() -> {valid}")
print(f"Reason: {msg}")

print("\nFull chain so far:")
chain.print_chain()


STEP 5: VALIDATE THE WHOLE CHAIN
is_chain_valid() -> True
Reason: Chain is valid: all hashes link correctly and every signature verifies.

Full chain so far:
--- Block 0 ---
  Timestamp     : 1788955663.489944
  Validator     : genesis
  Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000
  Hash          : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Signature     : genesis
  Transactions  : 0
--- Block 1 ---
  Timestamp     : 1788955663.5420365
  Validator     : validator_A
  Previous Hash : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Hash          : 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  Signature     : 3045022100805b8a182e217c...
  Transactions  : 1
    - {'event': 'PROPERTY_CREATED', 'property_id': 1, 'name': 'London Apartment', 'initial_value': 500000, 'total_shares': 3000, 'submitted_at': 1788955663.5416627}
--- Block 2 ---
  Timestamp     : 1788955663.5717082
  Validator     : 

4.5 Scenario 5 : tampering detection

In [12]:

# ======================================================================
banner("STEP 6: TAMPER WITH BLOCK #2's DATA — WATCH VALIDATION CATCH IT")
# ======================================================================
print(f"Original transaction in block #2: {block2.transactions[0]}")
print(f"Original block #2 hash (stored):  {block2.hash}\n")

print("Now we directly mutate the transaction WITHOUT recalculating the hash —")
print("simulating an attacker editing stored data after the fact.\n")
original_amount = block2.transactions[0]["amount"]
block2.transactions[0]["amount"] = 90_000
print(f"Tampered transaction in block #2: {block2.transactions[0]}")

recomputed_hash = block2.compute_hash()
print(f"\nRecomputing the hash from the NEW (tampered) content gives:")
print(f"  {recomputed_hash}")
print(f"But the block's STORED hash is still:")
print(f"  {block2.hash}")
print(f"These no longer match — that mismatch is exactly what is_chain_valid() checks.\n")

# ======================================================================
banner(": VALIDATing THE WHOLE CHAIN ")
# ======================================================================
valid, msg = chain.is_chain_valid()
print(f"is_chain_valid() -> {valid}")
print(f"Reason: {msg}")
# repair for the next steps
block2.transactions[0]["amount"] = original_amount
valid, msg = chain.is_chain_valid()
print(f"\n(Repaired back to the original value — is_chain_valid() -> {valid} again)")


STEP 6: TAMPER WITH BLOCK #2's DATA — WATCH VALIDATION CATCH IT
Original transaction in block #2: {'event': 'SHARE_TRANSFERRED', 'property_id': 1, 'from': '0xAlice', 'to': '0xBob', 'amount': 100, 'submitted_at': 1788955663.571571}
Original block #2 hash (stored):  838eeff7543497d857a856f19bd7cdb31ba7298b21eec872488d2131fc6e6eb8

Now we directly mutate the transaction WITHOUT recalculating the hash —
simulating an attacker editing stored data after the fact.

Tampered transaction in block #2: {'event': 'SHARE_TRANSFERRED', 'property_id': 1, 'from': '0xAlice', 'to': '0xBob', 'amount': 90000, 'submitted_at': 1788955663.571571}

Recomputing the hash from the NEW (tampered) content gives:
  4e3649df39836d5c6fca4d35e9f8c4bd307b6a3a19b6b7e21d3c8b88c343ba48
But the block's STORED hash is still:
  838eeff7543497d857a856f19bd7cdb31ba7298b21eec872488d2131fc6e6eb8
These no longer match — that mismatch is exactly what is_chain_valid() checks.


: VALIDATing THE WHOLE CHAIN 
is_chain_valid() -> Fal

In [13]:
# ======================================================================
banner(": VALIDATing THE WHOLE CHAIN ")
# ======================================================================

# checking the chain validation for net examples and printing the chain
#to check the stored hash is coreect cuz remember the same datat genrate the same hash

valid, msg = chain.is_chain_valid()
print(f"is_chain_valid() -> {valid}")
chain.print_chain()


: VALIDATing THE WHOLE CHAIN 
is_chain_valid() -> True
--- Block 0 ---
  Timestamp     : 1788955663.489944
  Validator     : genesis
  Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000
  Hash          : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Signature     : genesis
  Transactions  : 0
--- Block 1 ---
  Timestamp     : 1788955663.5420365
  Validator     : validator_A
  Previous Hash : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Hash          : 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  Signature     : 3045022100805b8a182e217c...
  Transactions  : 1
    - {'event': 'PROPERTY_CREATED', 'property_id': 1, 'name': 'London Apartment', 'initial_value': 500000, 'total_shares': 3000, 'submitted_at': 1788955663.5416627}
--- Block 2 ---
  Timestamp     : 1788955663.5717082
  Validator     : validator_B
  Previous Hash : 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  Hash  

4.6 Scenario 6 Unauthorized validator rejected (PermissionError)

In [14]:
# ======================================================================
banner("STEP 7: AN UNAUTHORIZED VALIDATOR TRIES TO MINE A BLOCK")
# ======================================================================
attacker = PoAValidator("AttackerValidator")
print(f"AttackerValidator's public key: {attacker.public_key_hex()}")
print(f"Is 'AttackerValidator' registered on the chain? "
      f"{chain.is_authorized('AttackerValidator')}\n")

chain.add_transaction({"event": "HEDGE_PURCHASED", "investor": "Investor_X", "premium": 42})
print("Attempting chain.mine_block(attacker)...\n")
try:
    chain.mine_block(attacker)
    print("ERROR: this should never print — the block should have been rejected.")
except PermissionError as e:
    print(f"PermissionError raised: {e}")
    print(f"Chain length is still {len(chain.chain)} — the block was NOT added.")

chain.pending_transactions = []  # clean up the rejected attempt's leftover transaction


STEP 7: AN UNAUTHORIZED VALIDATOR TRIES TO MINE A BLOCK
AttackerValidator's public key: 0468cc7a1765e89ec694febc07f7a942a5b7f5a04d96f0db4a85c4165fb29b75d392f72d08b861443ebfae8488f95fca4e13c9ed1d0d28c4c0e97ff3809996fca6
Is 'AttackerValidator' registered on the chain? False

Attempting chain.mine_block(attacker)...

PermissionError raised: 'AttackerValidator' is not an authorized PoA validator; block rejected.
Chain length is still 3 — the block was NOT added.


4.7 Scenraio 7 : Forged signature detection (impostor with a different private key)

In [15]:
# ======================================================================
banner("STEP 8: FORGE A SIGNATURE — SAME NAME, DIFFERENT PRIVATE KEY")
# ======================================================================
chain.add_transaction({"event": "HEDGE_SETTLED", "hedge_id": 1_000_001, "payout": 0})
block3 = chain.mine_block(validator_a)
print(f"block3 mined and signed normally by validator_A.")
print(f"  Real signature: {block3.signature[:40]}...\n")

impostor = PoAValidator("validator_A")  # SAME claimed id, DIFFERENT keypair
print(f"An impostor generates their OWN keypair, but claims to be 'validator_A':")
print(f"  Real validator_A's public key:     {validator_a.public_key_hex()}")
print(f"  Impostor's public key:             {impostor.public_key_hex()}")
print(f"  (Different, even though both claim to be 'validator_A')\n")

forged_signature = impostor.sign(block3.hash.encode())
print(f"Impostor signs block3's hash with THEIR private key:")
print(f"  Forged signature: {forged_signature[:40]}...\n")

print("Substituting the forged signature onto block3, then re-validating:")
original_signature = block3.signature
block3.signature = forged_signature

is_valid_forgery = verify_signature(
    chain.validator_public_keys["validator_A"], block3.hash.encode(), forged_signature
)
print(f"\nDoes the forged signature verify against the REAL registered")
print(f"public key for 'validator_A'? {is_valid_forgery}")

valid, msg = chain.is_chain_valid()
print(f"\nis_chain_valid() -> {valid}")
print(f"Reason: {msg}")

block3.signature = original_signature  # repair
valid, msg = chain.is_chain_valid()
print(f"\n(Repaired back to the real signature — is_chain_valid() -> {valid} again)")


STEP 8: FORGE A SIGNATURE — SAME NAME, DIFFERENT PRIVATE KEY
block3 mined and signed normally by validator_A.
  Real signature: 304402204dfd9eefcaadeba824d4e8c8bcd509b4...

An impostor generates their OWN keypair, but claims to be 'validator_A':
  Real validator_A's public key:     0455460516a4cc3935c808fdfeafdd5257b5e58d99efd7da5fabd61f48065dd5138ee8c29f93ea49a4a8c2811e7e2306bea0e141a7c8ab03e228ebb84044dd1fc1
  Impostor's public key:             04ace3fc6e424b1665b0acf3bb9311d54d6d0bd74addaa6864108598e25b3048ef1419ec51ea27cf858d4aa842241dffcddbcb40af56a35a87f7cb91a207f3c3c4
  (Different, even though both claim to be 'validator_A')

Impostor signs block3's hash with THEIR private key:
  Forged signature: 30450221009b51a1019ca8c728989a0cfba7cb31...

Substituting the forged signature onto block3, then re-validating:

Does the forged signature verify against the REAL registered
public key for 'validator_A'? False

is_chain_valid() -> False
Reason: Block 3's signature does not verify agai

4.8 : Blockchain testing results

In [16]:
# ======================================================================
banner("DONE — final chain state")
# ======================================================================
chain.print_chain()
print(f"\nFinal validation: {chain.is_chain_valid()}")


DONE — final chain state
--- Block 0 ---
  Timestamp     : 1788955663.489944
  Validator     : genesis
  Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000
  Hash          : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Signature     : genesis
  Transactions  : 0
--- Block 1 ---
  Timestamp     : 1788955663.5420365
  Validator     : validator_A
  Previous Hash : 60868ba4cb14aa79426692fa19880cc366638e7e3ac68b4bea3e4d3645ba5c11
  Hash          : 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  Signature     : 3045022100805b8a182e217c...
  Transactions  : 1
    - {'event': 'PROPERTY_CREATED', 'property_id': 1, 'name': 'London Apartment', 'initial_value': 500000, 'total_shares': 3000, 'submitted_at': 1788955663.5416627}
--- Block 2 ---
  Timestamp     : 1788955663.5717082
  Validator     : validator_B
  Previous Hash : 986f57eeceb81b6ba36f1defb83271683d181c199a7a56220fed677a8fdd25e8
  Hash          : 838eeff7543497d857a8